In [1]:
import os
import pandas as pd
from pathlib import Path

from met_council_wrangler import CubeTransit
from met_council_wrangler import MetCouncil_Parameters
from met_council_wrangler import metcouncil_transit
from cube_wrangler import StandardTransit
from cube_wrangler import Parameters
from cube_wrangler import Project

import network_wrangler
from network_wrangler import Scenario
from network_wrangler import load_roadway
from network_wrangler import load_transit
from network_wrangler import create_scenario
from network_wrangler.transit import write_transit

In [2]:
network_wrangler.setup_logging()

In [3]:
%reload_ext autoreload
%autoreload 2

### I/O

In [4]:
input_transit_dir = r"Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\transit_lin"
input_scenario_dir = r"Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\Networks\standard_networks\v02base\standard_networks"

output_transit_dir = os.path.join(r"Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\Networks\standard_transit")
project_card_dir = os.path.join(r"Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\project_card", "transit_project_cards")

metcouncil_wrangler_dir = os.path.join(r"Z:\Met_Council\Yue_temp\metcouncil\met_council_wrangler")
cube_wrangler_dir = os.path.join(r"Z:\Met_Council\Yue_temp\metcouncil\cube_wrangler")

In [5]:
metcouncil_parameters = MetCouncil_Parameters(
    metcouncil_wrangler_base_dir=metcouncil_wrangler_dir,
    cube_wrangler_base_dir=cube_wrangler_dir
)

2024-10-29 22:47:26, INFO: cube_wrangler base directory set as: Z:\Met_Council\Yue_temp\metcouncil\cube_wrangler
2024-10-29 22:47:26, INFO: MetCouncil Wrangler base directory set as: Z:\Met_Council\Yue_temp\metcouncil\met_council_wrangler


### Create Transit Project Card
Compare two transit.lin files and write out a yaml file with transit changes.

In [6]:
base_transit_source=os.path.join(input_transit_dir, "transit_rail.lin")
build_transit_source=os.path.join(input_transit_dir, "transit_rail_new.lin")
transit_shape_crosswalk_file=os.path.join(input_transit_dir,"line_name_xwalk.csv")

base_transit_network = CubeTransit.create_from_cube(
    transit_source = base_transit_source, 
    parameters = metcouncil_parameters,
    transit_shape_crosswalk_file = transit_shape_crosswalk_file,
    model_shape_id_column = "shp_index"
)

build_transit_network = CubeTransit.create_from_cube(
    transit_source = build_transit_source, 
    parameters = metcouncil_parameters,
    transit_shape_crosswalk_file = transit_shape_crosswalk_file, 
    model_shape_id_column = "shp_index"   
)

transit_project = Project.create_project(
    base_transit_network=base_transit_network,
    build_transit_network=build_transit_network,
    parameters=metcouncil_parameters,
)

transit_project.write_project_card(
    Path(os.path.join(project_card_dir, "test.yml"))
)

2024-10-29 22:47:27, INFO: Reading transit shape crosswalk file: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\transit_lin\line_name_xwalk.csv
2024-10-29 22:47:27, INFO: Will convert model shape id shp_index to standard shape_id
Creating a new Cube Transit instance
2024-10-29 22:47:27, DEBUG: Creating a new Cube Transit instance
2024-10-29 22:47:27, INFO: cube_wrangler base directory set as: Z:\Met_Council\Yue_temp\metcouncil\cube_wrangler
2024-10-29 22:47:27, INFO: MetCouncil Wrangler base directory set as: Z:\Met_Council\Yue_temp\metcouncil\met_council_wrangler
reading: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\transit_lin\transit_rail.lin
2024-10-29 22:47:27, DEBUG: reading transit source: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\transit_lin\transit_rail.lin
2024-10-29 22:47:30, DEBUG: finished parsing cube line file
2024-10-29 22:47:31, DEBUG: Added lines to CubeTran

### User to fill in the metadata in the result transit project card yaml file, i.e., the `project` name in the card file

### Apply all project cards in a directory

In [7]:
link_file = os.path.join(input_scenario_dir, "roadway", "v02base_link.json")
node_file = os.path.join(input_scenario_dir, "roadway", "v02base_node.geojson")
shape_file = os.path.join(input_scenario_dir, "roadway", "v02base_shape.geojson")


roadway_net = load_roadway(
    links_file=link_file,
    nodes_file=node_file,
    shapes_file=shape_file,
)

2024-10-29 22:48:24, DEBUG: Reading nodes from Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\Networks\standard_networks\v02base\standard_networks\roadway\v02base_node.geojson.
2024-10-29 22:48:24, DEBUG: Estimated read time: 4 seconds.
2024-10-29 22:48:43, DEBUG: Read 414282 nodes from file in 19.3.
2024-10-29 22:48:43, DEBUG: Turning node data into official nodes_df
2024-10-29 22:48:50, INFO: Read 414282 nodes from Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\Networks\standard_networks\v02base\standard_networks\roadway\v02base_node.geojson in 26.17.
2024-10-29 22:48:50, INFO: Reading links from Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\Networks\standard_networks\v02base\standard_networks\roadway\v02base_link.json.
2024-10-29 22:48:50, DEBUG: Estimated read time: 4 minutes.
2024-10-29 22:50:00, DEBUG: Read 1061756 links in 69.56.
2024-10-29 22:50:00, DEBUG: Creating 1061756 

In [8]:
transit_net = load_transit(os.path.join(input_scenario_dir, "transit"))

2024-10-29 22:51:33, INFO: Reading GTFS feed tables from Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\Networks\standard_networks\v02base\standard_networks\transit
2024-10-29 22:51:33, DEBUG: ...reading Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\Networks\standard_networks\v02base\standard_networks\transit\v02base_frequencies.txt.
2024-10-29 22:51:33, DEBUG: ...reading Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\Networks\standard_networks\v02base\standard_networks\transit\v02base_routes.txt.
2024-10-29 22:51:33, DEBUG: ...reading Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\Networks\standard_networks\v02base\standard_networks\transit\v02base_shapes.txt.
2024-10-29 22:51:33, DEBUG: ...reading Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\Networks\standard_networks\v02base\standard_networks\transit\v02base

c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\network_wrangler\transit\io.py:81: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(file)


2024-10-29 22:51:33, INFO: Initializing frequencies
2024-10-29 22:51:33, DEBUG: Validating + coercing value to frequencies
2024-10-29 22:51:34, DEBUG: PK table trips for specified FK                     frequencies.trip_id not in <class 'network_wrangler.transit.feed.feed.Feed'>-                      skipping validation.
2024-10-29 22:51:34, INFO: Initializing routes
2024-10-29 22:51:34, DEBUG: Validating + coercing value to routes
2024-10-29 22:51:34, DEBUG: PK table agencies for specified FK                     routes.agency_id not in table list - skipping validation.
2024-10-29 22:51:34, DEBUG: Referencing table trips not yet set in                      <class 'network_wrangler.transit.feed.feed.Feed'> - skipping fk validation.
2024-10-29 22:51:34, INFO: Initializing shapes
2024-10-29 22:51:34, DEBUG: Validating + coercing value to shapes
2024-10-29 22:51:34, INFO: Initializing stops
2024-10-29 22:51:34, DEBUG: Validating + coercing value to stops
2024-10-29 22:51:34, DEBUG: PK tabl

c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\pandera\engines\pandas_engine.py:873: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  col = to_datetime_fn(col, **self.to_datetime_kwargs)
c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\pandera\engines\pandas_engine.py:873: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  col = to_datetime_fn(col, **self.to_datetime_kwargs)


In [9]:
base_scenario = {"road_net": roadway_net, "transit_net": transit_net}
version_0_scenario = create_scenario(base_scenario = base_scenario)

2024-10-29 22:51:35, INFO: Creating Scenario
2024-10-29 22:51:35, WARNING: Base_scenario doesn't contain ['road_net', 'transit_net', 'applied_projects', 'conflicts']
2024-10-29 22:51:37, DEBUG: Validating + coercing value to frequencies
2024-10-29 22:51:37, DEBUG: PK table trips for specified FK                     frequencies.trip_id not in <class 'network_wrangler.transit.feed.feed.Feed'>-                      skipping validation.
2024-10-29 22:51:37, DEBUG: Validating + coercing value to routes
2024-10-29 22:51:37, DEBUG: PK table agencies for specified FK                     routes.agency_id not in table list - skipping validation.
2024-10-29 22:51:37, DEBUG: Referencing table trips not yet set in                      <class 'network_wrangler.transit.feed.feed.Feed'> - skipping fk validation.
2024-10-29 22:51:37, DEBUG: Validating + coercing value to shapes
2024-10-29 22:51:37, DEBUG: Validating + coercing value to stops
2024-10-29 22:51:37, DEBUG: PK table stops for specified FK  

In [10]:
version_01_scenario = create_scenario(
    base_scenario = version_0_scenario,
    project_card_filepath = project_card_dir,
)
version_01_scenario.transit_net.road_net = version_01_scenario.road_net

2024-10-29 22:51:43, INFO: Creating Scenario
2024-10-29 22:51:43, DEBUG: Validating + coercing value to frequencies
2024-10-29 22:51:44, DEBUG: PK table trips for specified FK                     frequencies.trip_id not in <class 'network_wrangler.transit.feed.feed.Feed'>-                      skipping validation.
2024-10-29 22:51:44, DEBUG: Validating + coercing value to routes
2024-10-29 22:51:44, DEBUG: PK table agencies for specified FK                     routes.agency_id not in table list - skipping validation.
2024-10-29 22:51:44, DEBUG: Referencing table trips not yet set in                      <class 'network_wrangler.transit.feed.feed.Feed'> - skipping fk validation.
2024-10-29 22:51:44, DEBUG: Validating + coercing value to shapes
2024-10-29 22:51:44, DEBUG: Validating + coercing value to stops
2024-10-29 22:51:44, DEBUG: PK table stops for specified FK                     stops.parent_station not in <class 'network_wrangler.transit.feed.feed.Feed'>-                      sk

In [11]:
version_01_scenario.apply_all_projects()

2024-10-29 22:51:56, DEBUG: Ordered Projects: 
deque(['update transit', 'delete transit', 'add transit'])
2024-10-29 22:51:56, INFO: Applying update transit from file:                            Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\project_card\transit_project_cards\update_transit.yml
2024-10-29 22:51:56, DEBUG: types: ['transit_routing_change', 'transit_property_change', 'transit_property_change', 'transit_property_change', 'transit_property_change']
2024-10-29 22:51:56, DEBUG: type: multiple
2024-10-29 22:51:56, DEBUG: - applying subproject: transit_routing_change
2024-10-29 22:51:56, DEBUG: Performing selection from key: 5a2e6469c72d1ef8c2763224d35abe27152cb9ff
2024-10-29 22:51:57, DEBUG: SELECT DICT - before Validation: 
{'trip_properties': {'shape_id': ['65010034'], 'direction_id': 0, 'route_id': ['10']}, 'timespans': [['3:00', '6:00'], ['6:00', '10:00'], ['10:00', '15:00'], ['15:00', '19:00'], ['19:00', '3:00']]}
2024-10-29 22:51:57, D

c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\network_wrangler\utils\time.py:186: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  end_time_s.loc[orig_df["end_time"] < orig_df["start_time"]] += pd.Timedelta(days=1)
c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\network_wrangler\utils\time.py:186: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  end_time_s.loc[orig_df["end_time"] < orig_df["start_time"]] += pd.Timedelta(days=1)
c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\network_wrangler\utils\time.py:186: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a 

2024-10-29 22:52:02, DEBUG: Updating shapes and trips for shape_id: 65010034
2024-10-29 22:52:02, DEBUG: Creating new shape for shape_id: 65010034
2024-10-29 22:52:03, DEBUG: Validating + coercing value to shapes
2024-10-29 22:52:03, DEBUG: Validating + coercing value to shapes
2024-10-29 22:52:03, DEBUG: Validating + coercing value to trips
2024-10-29 22:52:03, DEBUG: Updating stops for transit routing change.
2024-10-29 22:52:03, DEBUG: Validating + coercing value to stops
2024-10-29 22:52:04, DEBUG: Updating stop times for trip: 24251608-AUG23-MVS-BUS-Weekday-01
2024-10-29 22:52:04, DEBUG: Looking for stops near node_id: 20334
2024-10-29 22:52:04, DEBUG: Start/End nodes w/stops: 28234/18952
2024-10-29 22:52:04, DEBUG: Set stop nodes: [28234, 15868, 342998, 337252, 26998, 36105, 27464, 28427, 18966, 18952]
2024-10-29 22:52:04, DEBUG: Creating new stop times for trip: 24251608-AUG23-MVS-BUS-Weekday-01
2024-10-29 22:52:04, DEBUG: Deleting stop times for nodes: [20334]
2024-10-29 22:52:

c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\network_wrangler\utils\time.py:186: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  end_time_s.loc[orig_df["end_time"] < orig_df["start_time"]] += pd.Timedelta(days=1)
c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\network_wrangler\utils\time.py:186: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  end_time_s.loc[orig_df["end_time"] < orig_df["start_time"]] += pd.Timedelta(days=1)


2024-10-29 22:52:08, DEBUG: Performing selection from key: 0781d74e40316579a89cdd13e2b1cc733ca20b02
2024-10-29 22:52:08, DEBUG: SELECT DICT - before Validation: 
{'trip_properties': {'shape_id': ['65902005'], 'direction_id': 0, 'route_id': ['902']}, 'timespans': [['15:00', '19:00']]}
2024-10-29 22:52:08, DEBUG: ...created TransitSelection object: {'trip_properties': {'route_id': ['902'], 'direction_id': 0, 'shape_id': ['65902005']}, 'timespans': [['15:00', '19:00']]}
2024-10-29 22:52:08, DEBUG: Applying transit property change project.
2024-10-29 22:52:08, DEBUG: ...modifying headway_secs in frequencies.
2024-10-29 22:52:08, DEBUG: Building selection query
2024-10-29 22:52:08, DEBUG: Selection query: 
((shape_id.str.contains("65902005")) and direction_id==0 and (route_id.str.contains("902")))
2024-10-29 22:52:08, DEBUG: 5/1245 trips remain after         filtering to trip selection {'shape_id': ['65902005'], 'direction_id': 0, 'route_id': ['902']}
2024-10-29 22:52:08, DEBUG: # Trips aft

...

2024-10-29 22:52:10, DEBUG: Deleting service from feed.
2024-10-29 22:52:10, DEBUG: Validating + coercing value to stop_times
2024-10-29 22:52:14, DEBUG: Validating + coercing value to frequencies
2024-10-29 22:52:14, DEBUG: Validating + coercing value to trips
2024-10-29 22:52:16, INFO: Applying add transit from file:                            Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\project_card\transit_project_cards\add_transit.yml
2024-10-29 22:52:16, DEBUG: types: ['transit_route_addition']
2024-10-29 22:52:16, DEBUG: type: transit_route_addition
2024-10-29 22:52:16, DEBUG: - applying subproject: transit_route_addition
2024-10-29 22:52:16, DEBUG: Applying add transit route project.
2024-10-29 22:52:16, DEBUG: Adding route 2 to feed.
2024-10-29 22:52:16, DEBUG: Adding 2 trips for route abc.
2024-10-29 22:52:17, DEBUG: Adding 2 trips for route bbc.
2024-10-29 22:52:17, WARNING: Timespan is not in increasing order: ['19:00', '3:00'].         

c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\network_wrangler\utils\data.py:705: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat(dfs, **kwargs)
c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\network_wrangler\utils\data.py:705: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat(dfs, **kwargs)


2024-10-29 22:52:17, WARNING: Timespan is not in increasing order: ['19:00', '3:00'].            End time will be treated as next day.
2024-10-29 22:52:17, DEBUG: Validating + coercing value to routes
2024-10-29 22:52:17, DEBUG: PK table agencies for specified FK                     routes.agency_id not in table list - skipping validation.
2024-10-29 22:52:17, DEBUG: Validating + coercing value to shapes


c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\network_wrangler\utils\data.py:705: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat(dfs, **kwargs)
c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\network_wrangler\utils\data.py:705: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat(dfs, **kwargs)


2024-10-29 22:52:17, DEBUG: Validating + coercing value to trips
2024-10-29 22:52:17, DEBUG: Validating + coercing value to stops
2024-10-29 22:52:17, DEBUG: Validating + coercing value to stop_times
2024-10-29 22:52:17, DEBUG: Validating + coercing value to frequencies


In [12]:
version_01_scenario.applied_projects

['update transit', 'delete transit', 'add transit']

## Apply single project card
Can be used to apply single project card. Uncomment the four lines below to test it.

In [13]:
from projectcard import read_card
card_path = os.path.join(project_card_dir, "update_transit.yml")
card = read_card(card_path)
test = transit_net.apply(card, reference_road_net=roadway_net)

2024-10-29 22:52:21, DEBUG: - applying subproject: transit_routing_change
2024-10-29 22:52:21, DEBUG: Performing selection from key: 5a2e6469c72d1ef8c2763224d35abe27152cb9ff
2024-10-29 22:52:21, DEBUG: SELECT DICT - before Validation: 
{'trip_properties': {'shape_id': ['65010034'], 'direction_id': 0, 'route_id': ['10']}, 'timespans': [['3:00', '6:00'], ['6:00', '10:00'], ['10:00', '15:00'], ['15:00', '19:00'], ['19:00', '3:00']]}
2024-10-29 22:52:21, DEBUG: ...created TransitSelection object: {'trip_properties': {'route_id': ['10'], 'direction_id': 0, 'shape_id': ['65010034']}, 'timespans': [['3:00', '6:00'], ['6:00', '10:00'], ['10:00', '15:00'], ['15:00', '19:00'], ['19:00', '3:00']]}
2024-10-29 22:52:21, DEBUG: Applying transit routing change project.
2024-10-29 22:52:21, DEBUG: ...selection: {'trip_properties': {'shape_id': ['65010034'], 'direction_id': 0, 'route_id': ['10']}, 'timespans': [['3:00', '6:00'], ['6:00', '10:00'], ['10:00', '15:00'], ['15:00', '19:00'], ['19:00', '3:00

c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\network_wrangler\utils\time.py:186: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  end_time_s.loc[orig_df["end_time"] < orig_df["start_time"]] += pd.Timedelta(days=1)
c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\network_wrangler\utils\time.py:186: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  end_time_s.loc[orig_df["end_time"] < orig_df["start_time"]] += pd.Timedelta(days=1)
c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\network_wrangler\utils\time.py:186: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a 

2024-10-29 22:52:22, DEBUG: Updating shapes and trips for shape_id: 65010034
2024-10-29 22:52:22, DEBUG: Creating new shape for shape_id: 65010034
2024-10-29 22:52:23, DEBUG: Validating + coercing value to shapes
2024-10-29 22:52:23, DEBUG: Validating + coercing value to shapes
2024-10-29 22:52:23, DEBUG: Validating + coercing value to trips
2024-10-29 22:52:24, DEBUG: Updating stops for transit routing change.
2024-10-29 22:52:24, DEBUG: Validating + coercing value to stops
2024-10-29 22:52:24, DEBUG: Updating stop times for trip: 24251608-AUG23-MVS-BUS-Weekday-01
2024-10-29 22:52:24, DEBUG: Looking for stops near node_id: 20334
2024-10-29 22:52:24, DEBUG: Start/End nodes w/stops: 28234/18952
2024-10-29 22:52:24, DEBUG: Set stop nodes: [28234, 15868, 342998, 337252, 26998, 36105, 27464, 28427, 18966, 18952]
2024-10-29 22:52:24, DEBUG: Creating new stop times for trip: 24251608-AUG23-MVS-BUS-Weekday-01
2024-10-29 22:52:24, DEBUG: Deleting stop times for nodes: [20334]
2024-10-29 22:52:

c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\network_wrangler\utils\time.py:186: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  end_time_s.loc[orig_df["end_time"] < orig_df["start_time"]] += pd.Timedelta(days=1)
c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\network_wrangler\utils\time.py:186: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  end_time_s.loc[orig_df["end_time"] < orig_df["start_time"]] += pd.Timedelta(days=1)


2024-10-29 22:52:26, DEBUG: ...created TransitSelection object: {'trip_properties': {'route_id': ['902'], 'direction_id': 0, 'shape_id': ['65902005']}, 'timespans': [['15:00', '19:00']]}
2024-10-29 22:52:26, DEBUG: Applying transit property change project.
2024-10-29 22:52:26, DEBUG: ...modifying headway_secs in frequencies.
2024-10-29 22:52:26, DEBUG: Building selection query
2024-10-29 22:52:26, DEBUG: Selection query: 
((shape_id.str.contains("65902005")) and direction_id==0 and (route_id.str.contains("902")))
2024-10-29 22:52:26, DEBUG: 5/1245 trips remain after         filtering to trip selection {'shape_id': ['65902005'], 'direction_id': 0, 'route_id': ['902']}
2024-10-29 22:52:26, DEBUG: # Trips after trip property filter: 5
2024-10-29 22:52:26, DEBUG: 1/5 trips remain after         filtering to timespans [['15:00', '19:00']]
2024-10-29 22:52:26, DEBUG: 8    24209353-AUG23-RAIL-Weekday-01
Name: trip_id, dtype: object
2024-10-29 22:52:26, DEBUG: # Trips after timespans filter: 1


c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\network_wrangler\utils\time.py:186: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  end_time_s.loc[orig_df["end_time"] < orig_df["start_time"]] += pd.Timedelta(days=1)
c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\network_wrangler\utils\time.py:186: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  end_time_s.loc[orig_df["end_time"] < orig_df["start_time"]] += pd.Timedelta(days=1)


2024-10-29 22:52:30, DEBUG: ...created TransitSelection object: {'trip_properties': {'route_id': ['902'], 'direction_id': 0, 'shape_id': ['65902005']}, 'timespans': [['3:00', '6:00']]}
2024-10-29 22:52:30, DEBUG: Applying transit property change project.
2024-10-29 22:52:30, DEBUG: ...modifying headway_secs in frequencies.
2024-10-29 22:52:30, DEBUG: Building selection query
2024-10-29 22:52:30, DEBUG: Selection query: 
((shape_id.str.contains("65902005")) and direction_id==0 and (route_id.str.contains("902")))
2024-10-29 22:52:30, DEBUG: 5/1245 trips remain after         filtering to trip selection {'shape_id': ['65902005'], 'direction_id': 0, 'route_id': ['902']}
2024-10-29 22:52:30, DEBUG: # Trips after trip property filter: 5
2024-10-29 22:52:30, DEBUG: 1/5 trips remain after         filtering to timespans [['3:00', '6:00']]
2024-10-29 22:52:30, DEBUG: 5    24209317-AUG23-RAIL-Weekday-01
Name: trip_id, dtype: object
2024-10-29 22:52:30, DEBUG: # Trips after timespans filter: 1
2024

c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\network_wrangler\utils\time.py:186: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  end_time_s.loc[orig_df["end_time"] < orig_df["start_time"]] += pd.Timedelta(days=1)
c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\network_wrangler\utils\time.py:186: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  end_time_s.loc[orig_df["end_time"] < orig_df["start_time"]] += pd.Timedelta(days=1)


2024-10-29 22:52:31, DEBUG: ...created TransitSelection object: {'trip_properties': {'route_id': ['902'], 'direction_id': 0, 'shape_id': ['65902005']}, 'timespans': [['6:00', '10:00']]}
2024-10-29 22:52:31, DEBUG: Applying transit property change project.
2024-10-29 22:52:31, DEBUG: ...modifying headway_secs in frequencies.
2024-10-29 22:52:31, DEBUG: Building selection query
2024-10-29 22:52:31, DEBUG: Selection query: 
((shape_id.str.contains("65902005")) and direction_id==0 and (route_id.str.contains("902")))
2024-10-29 22:52:31, DEBUG: 5/1245 trips remain after         filtering to trip selection {'shape_id': ['65902005'], 'direction_id': 0, 'route_id': ['902']}
2024-10-29 22:52:31, DEBUG: # Trips after trip property filter: 5
2024-10-29 22:52:31, DEBUG: 1/5 trips remain after         filtering to timespans [['6:00', '10:00']]
2024-10-29 22:52:31, DEBUG: 6    24209322-AUG23-RAIL-Weekday-01
Name: trip_id, dtype: object
2024-10-29 22:52:31, DEBUG: # Trips after timespans filter: 1
20

c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\network_wrangler\utils\time.py:186: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  end_time_s.loc[orig_df["end_time"] < orig_df["start_time"]] += pd.Timedelta(days=1)
c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\network_wrangler\utils\time.py:186: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  end_time_s.loc[orig_df["end_time"] < orig_df["start_time"]] += pd.Timedelta(days=1)


### Write out new transit network in both cube and standard formats

In [14]:
write_transit(version_01_scenario.transit_net, file_format="txt", out_dir= output_transit_dir, overwrite=True)

2024-10-29 22:52:31, DEBUG: Writing to Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\Networks\standard_transit\frequencies.txt.
2024-10-29 22:52:31, DEBUG: Writing to Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\Networks\standard_transit\routes.txt.
2024-10-29 22:52:31, DEBUG: Writing to Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\Networks\standard_transit\shapes.txt.
2024-10-29 22:52:33, DEBUG: Writing to Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\Networks\standard_transit\stops.txt.
2024-10-29 22:52:33, DEBUG: Writing to Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\Networks\standard_transit\trips.txt.
2024-10-29 22:52:33, DEBUG: Writing to Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\Networks\standard_transit\stop_times.txt.
2024-10-29 22:52:34, INFO: Wrote 6 file

In [15]:
standard_transit_net = StandardTransit.fromTransitNetwork(version_01_scenario.transit_net, parameters=metcouncil_parameters)

standard_transit_net = metcouncil_transit.transit_standard_to_met_council_transit_network(
    transit_net = standard_transit_net,
    parameters = metcouncil_parameters,
    line_name_xwalk = os.path.join(output_transit_dir, "line_name_xwalk.csv")
)    

2024-10-29 22:52:35, INFO: cube_wrangler base directory set as: Z:\Met_Council\Yue_temp\metcouncil\cube_wrangler
2024-10-29 22:52:35, INFO: MetCouncil Wrangler base directory set as: Z:\Met_Council\Yue_temp\metcouncil\met_council_wrangler
No missing values found in column 'agency_raw_name'.
Missing values found and filled in column 'agency_raw_name'.
Missing values found and filled in column 'agency_raw_name'.
Missing values found and filled in column 'agency_raw_name'.
Missing values found and filled in column 'agency_raw_name'.
2024-10-29 22:52:37, INFO: Converting GTFS Standard Properties to MetCouncil's Cube Standard
2024-10-29 22:52:37, INFO: cube_wrangler base directory set as: Z:\Met_Council\Yue_temp\metcouncil\cube_wrangler
2024-10-29 22:52:37, INFO: MetCouncil Wrangler base directory set as: Z:\Met_Council\Yue_temp\metcouncil\met_council_wrangler


\\dcclda00dat02.corp.pbwan.net\sag\projects\met_council\31000743a\task 1 consolidate architecture\software\met_council_wrangler\met_council_wrangler\metcouncil_transit.py:1175: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[column_name].fillna(default_value, inplace=True)
\\dcclda00dat02.corp.pbwan.net\sag\projects\met_council\31000743a\task 1 consolidate architecture\software\met_council_wrangler\met_council_wrangler\metcouncil_transit.py:1308: FutureWarning: The default of observed=False is deprecated and will be changed t

2024-10-29 22:52:37, DEBUG: Validating + coercing value to shapes
2024-10-29 22:52:38, DEBUG: Validating + coercing value to shapes
2024-10-29 22:52:38, DEBUG: Validating + coercing value to shapes
2024-10-29 22:52:38, DEBUG: Validating + coercing value to shapes
2024-10-29 22:52:39, DEBUG: Validating + coercing value to shapes
2024-10-29 22:52:39, DEBUG: Validating + coercing value to shapes
2024-10-29 22:52:40, DEBUG: Validating + coercing value to shapes
2024-10-29 22:52:40, DEBUG: Validating + coercing value to shapes
2024-10-29 22:52:40, DEBUG: Validating + coercing value to shapes
2024-10-29 22:52:41, DEBUG: Validating + coercing value to shapes
2024-10-29 22:52:41, DEBUG: Validating + coercing value to shapes
2024-10-29 22:52:42, DEBUG: Validating + coercing value to shapes
2024-10-29 22:52:42, DEBUG: Validating + coercing value to shapes
2024-10-29 22:52:42, DEBUG: Validating + coercing value to shapes
2024-10-29 22:52:43, DEBUG: Validating + coercing value to shapes
2024-10-29

In [16]:
standard_transit_net.write_as_cube_lin(outpath = os.path.join(output_transit_dir, "transit.lin"))